In [1]:
import torch
import torch.nn as nn
from transformers import BertModel, DistilBertModel
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import duckdb as db
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
import re
from torch.cuda.amp import autocast
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [6]:
load_dotenv()

db_path = os.getenv("DATABASE_PATH")

conn = db.connect(db_path)

df = conn.sql("""select * from processed_news
""").df()


In [5]:
def remove_leaks(text):
    text = re.sub(r'^.*?\(Reuters\)\s*-', '', text)
    leak_words = r'\b(reuters|via|twitter|video|watch|breaking|shared)\b'
    text = re.sub(leak_words, '', text, flags=re.IGNORECASE)
    return text

df['full_text'] = df['text'] + df['title']
df['full_text'] = df['full_text'].apply(remove_leaks)
X = df['full_text']
y = df['is_fake']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [7]:
df

,title,text,subject,is_fake,date,word_count,character_count,average_word_length,title_word_count,title_character_count,exclamation_count,uppercase_ratio,url_count
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,1,2017-12-31,504,2893,5.740079,13,79,6,0.064276,0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,1,2017-12-31,316,1898,6.006329,8,69,0,0.056884,0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,1,2017-12-30,609,3597,5.906404,15,90,2,0.111959,1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,1,2017-12-29,474,2774,5.852321,15,78,0,0.058377,4
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,1,2017-12-25,422,2346,5.559242,11,70,0,0.033636,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39094,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,0,2017-08-22,473,2821,5.964059,10,61,0,0.051716,0
39095,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,0,2017-08-22,124,800,6.451613,7,52,0,0.060465,0
39096,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,0,2017-08-22,326,1950,5.981595,7,49,0,0.034794,0
39097,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,0,2017-08-22,205,1199,5.848780,9,61,0,0.057292,0


In [7]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, model_name = "distilbert-base-uncased", max_len = 128):
        super().__init__()
        self.texts = texts
        self.labels = labels
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        tokens = self.tokenizer(text, add_special_tokens=True, truncation = True, padding = "max_length", max_length=self.max_len, return_tensors='pt')

        item = {key: tensor.squeeze(0) for key, tensor in tokens.items()}

        item['label'] = torch.tensor(label, dtype=torch.long)

        return item

In [8]:
train_dataset = FakeNewsDataset(X_train, y_train)
test_dataset = FakeNewsDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
class BertFakeNewsClassifier(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", drop_rate=0.3):
        super().__init__()
        self.model = DistilBertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(drop_rate)
        self.classifier = nn.Linear(self.model.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids, attention_mask)
        last_hidden_state = outputs.last_hidden_state
        cls_output = last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        x = self.classifier(x)

        return x

In [10]:
model = BertFakeNewsClassifier()
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.3)
scaler = torch.amp.GradScaler("cuda")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
patience = 2
patience_counter = 0
best_val_loss = float('inf')
for epoch in range(2):
    model.train()
    train_loss = 0
    for index, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        with torch.amp.autocast(device_type='cuda'):
          preds = model(input_ids, attention_mask)
          loss = criterion(preds, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        train_loss += loss.item()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        if index % 20 == 0:
          print(f"Batch {index}, train loss {train_loss:.4f}")

    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    test_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            preds = model(input_ids, attention_mask)

            loss = criterion(preds, labels)

            test_loss += loss.item()

            pred = torch.argmax(preds, dim=1)
            probs = torch.softmax(preds, dim=1)[:,1]

            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_train_loss = train_loss/len(train_loader)
    avg_test_loss = test_loss / len(test_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs)
    scheduler.step(test_loss)

    if avg_test_loss < best_val_loss:
      best_val_loss = avg_test_loss
      patience_counter = 0
      torch.save(model.state_dict(), 'best_model.pt')
    else:
      patience_counter += 1
      if patience_counter > patience:
        print(f" Early stopping was activated on {epoch}! End.")
        break
    print(f'{epoch} train loss {avg_train_loss} test loss {avg_test_loss} accuracy {accuracy} f1 score {f1} roc auc score {roc_auc}')

Batch 0, train loss 0.0268
Batch 20, train loss 2.5830
Batch 40, train loss 4.4765
Batch 60, train loss 6.1926
Batch 80, train loss 7.8044
Batch 100, train loss 8.5857
Batch 120, train loss 9.7938
Batch 140, train loss 11.2808
Batch 160, train loss 12.6828
Batch 180, train loss 13.9095
Batch 200, train loss 15.4454
Batch 220, train loss 17.1312
Batch 240, train loss 18.0871
Batch 260, train loss 18.8680
Batch 280, train loss 19.4493
Batch 300, train loss 20.2970
Batch 320, train loss 21.7380
Batch 340, train loss 22.3844
Batch 360, train loss 23.1048
Batch 380, train loss 23.2079
Batch 400, train loss 24.2755
Batch 420, train loss 24.6198
Batch 440, train loss 26.4196
Batch 460, train loss 26.7349
Batch 480, train loss 27.6524
Batch 500, train loss 28.9914
Batch 520, train loss 29.3090
Batch 540, train loss 29.9592
Batch 560, train loss 30.8665
Batch 580, train loss 31.3358
Batch 600, train loss 31.4309
Batch 620, train loss 31.5404
Batch 640, train loss 32.2593
Batch 660, train loss 3